# Sanity check: CUDA / Intel Extension for TensorFlow / CPU fallback

This notebook checks:
- CUDA availability in TensorFlow
- Intel Extension for TensorFlow (IETF/ITEX) availability
- Automatic fallback to CPU when no supported accelerator is available

It does **not** require either CUDA or ITEX to be installed; it will still run and report a CPU fallback.

Install Intel XPU:
- python 3.11
- pip install tensorflow==2.15.1
- pip install --upgrade intel-extension-for-tensorflow[xpu]


In [1]:
import importlib.util
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import tensorflow as tf
except Exception as e:
    raise RuntimeError(
        "TensorFlow is not importable in this kernel environment. "
        "Install TensorFlow first, then rerun this notebook."
    ) from e

print("TensorFlow:", tf.__version__)

Python: 3.11.14
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39


2026-03-04 06:38:11.796684: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 06:38:11.903889: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 06:38:12.369150: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-04 06:38:12.369409: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-04 06:38:12.472156: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

RuntimeError: TensorFlow is not importable in this kernel environment. Install TensorFlow first, then rerun this notebook.

In [ ]:
def _safe_list_physical(device_type: str):
    try:
        return tf.config.list_physical_devices(device_type)
    except Exception:
        return []


def check_cuda_status():
    gpus = _safe_list_physical("GPU")
    built_with_cuda = False
    try:
        built_with_cuda = bool(tf.test.is_built_with_cuda())
    except Exception:
        built_with_cuda = False

    return {
        "built_with_cuda": built_with_cuda,
        "num_visible_gpu": len(gpus),
        "gpu_devices": [d.name for d in gpus],
        "cuda_available": len(gpus) > 0,
    }


cuda_status = check_cuda_status()
print("CUDA build:", cuda_status["built_with_cuda"])
print("Visible GPU count:", cuda_status["num_visible_gpu"])
if cuda_status["gpu_devices"]:
    for name in cuda_status["gpu_devices"]:
        print(" -", name)
else:
    print("No TensorFlow GPU devices detected.")

In [ ]:
def check_intel_extension_status():
    status = {
        "module_found": False,
        "import_ok": False,
        "itex_version": None,
        "xpu_devices": [],
        "notes": [],
    }

    spec = importlib.util.find_spec("intel_extension_for_tensorflow")
    status["module_found"] = spec is not None

    if not status["module_found"]:
        status["notes"].append(
            "intel_extension_for_tensorflow is not installed in this environment."
        )
        return status

    try:
        import intel_extension_for_tensorflow as itex

        status["import_ok"] = True
        status["itex_version"] = getattr(itex, "__version__", "unknown")

        # On supported setups, ITEX may expose XPU devices to TensorFlow.
        xpus = _safe_list_physical("XPU")
        status["xpu_devices"] = [d.name for d in xpus]

        if not status["xpu_devices"]:
            status["notes"].append(
                "ITEX imported, but no XPU devices are visible to TensorFlow."
            )
    except Exception as e:
        status["notes"].append(
            "ITEX package exists but failed to import. This often indicates a "
            "TensorFlow/ITEX version mismatch."
        )
        status["notes"].append(f"Import error: {e}")

    return status


itex_status = check_intel_extension_status()
print("ITEX installed:", itex_status["module_found"])
print("ITEX importable:", itex_status["import_ok"])
print("ITEX version:", itex_status["itex_version"])
print("XPU devices:", len(itex_status["xpu_devices"]))
for name in itex_status["xpu_devices"]:
    print(" -", name)
for note in itex_status["notes"]:
    print("[note]", note)

In [ ]:
# Device preference:
# 1) CUDA GPU (if visible)
# 2) Intel XPU via ITEX (if importable and visible)
# 3) CPU fallback
if cuda_status["cuda_available"]:
    selected_backend = "cuda"
    selected_device = "GPU"
elif itex_status["import_ok"] and len(itex_status["xpu_devices"]) > 0:
    selected_backend = "itex"
    selected_device = "XPU"
else:
    selected_backend = "cpu"
    selected_device = "CPU"

print("\n=== Final decision ===")
print("selected_backend:", selected_backend)
print("selected_device:", selected_device)
if selected_backend == "cpu":
    print(
        "Falling back to CPU. Reason: no visible CUDA GPU and/or no usable ITEX setup."
    )

# Optional quick tensor op to verify runtime path is functional.
with tf.device(f"/{selected_device}:0"):
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
    c = tf.matmul(a, b)
print("MatMul result:\n", c.numpy())